# Practice Dataset Generation
**10,000 rows · 1% fraud rate (100 cases) · No label leakage**

Run this cell once. It saves `practice_dataset.csv` to `e:/project/data/raw/`.

In [ ]:
import pandas as pd
import numpy as np
import uuid
from datetime import datetime, timedelta

np.random.seed(42)

# ── PARAMETERS ──────────────────────────────────────────────────────────────
N_TOTAL  = 10_000
N_FRAUD  = 100        # 1% fraud rate
N_LEGIT  = 9_900

BUSINESS_TYPES = ['Exporter', 'Importer', 'IT Services', 'EdTech', 'SaaS', 'SME']
BIZ_WEIGHTS    = [0.25, 0.20, 0.20, 0.15, 0.10, 0.10]

CORRIDORS   = ['USA', 'UK', 'UAE', 'Singapore', 'Germany', 'Australia', 'Canada', 'Japan']
COR_WEIGHTS = [0.25, 0.20, 0.18, 0.12, 0.08, 0.07, 0.06, 0.04]

RAILS        = ['SWIFT', 'Fedwire', 'CHIPS', 'SEPA', 'CHAPS']
RAIL_WEIGHTS = [0.60, 0.15, 0.10, 0.10, 0.05]

FRAUD_TYPES = [
    'TBML_over_invoicing', 'TBML_under_invoicing', 'round_tripping',
    'ATO', 'structuring', 'dormant_account', 'purpose_code_abuse'
]

START_DATE = datetime(2026, 1, 1)

# ── HELPER ───────────────────────────────────────────────────────────────────
def make_timestamps(hours, n):
    days = np.random.randint(0, 180, size=n)
    mins = np.random.randint(0,  60, size=n)
    return [START_DATE + timedelta(days=int(d), hours=int(h), minutes=int(m))
            for d, h, m in zip(days, hours, mins)]

# ── LEGITIMATE TRANSACTIONS (9,900 rows) ─────────────────────────────────────
legit_amounts = np.clip(
    np.random.lognormal(mean=9.9, sigma=0.75, size=N_LEGIT), 1000, 200000
).round(2)

lh = np.array([.5,.3,.3,.3,.3,.5, 1,2,4,6,7,7, 6,7,7,6,5,4, 3,2,2,1.5,1,.8])
lh /= lh.sum()
legit_hours = np.random.choice(24, size=N_LEGIT, p=lh)

legit_df = pd.DataFrame({
    'transaction_id':      [str(uuid.uuid4())[:12] for _ in range(N_LEGIT)],
    'timestamp':           make_timestamps(legit_hours, N_LEGIT),
    'hour_of_day':         legit_hours,
    'day_of_week':         np.random.randint(0, 7, size=N_LEGIT),
    'month':               np.random.randint(1, 7, size=N_LEGIT),
    'amount_usd':          legit_amounts,
    'business_type':       np.random.choice(BUSINESS_TYPES, size=N_LEGIT, p=BIZ_WEIGHTS),
    'beneficiary_country': np.random.choice(CORRIDORS,      size=N_LEGIT, p=COR_WEIGHTS),
    'payment_rail':        np.random.choice(RAILS,          size=N_LEGIT, p=RAIL_WEIGHTS),
    'is_new_beneficiary':  np.random.choice([0, 1], size=N_LEGIT, p=[0.85, 0.15]),
    'is_fraud':            0,
    'fraud_type':          np.nan,
})

# ── FRAUD TRANSACTIONS (100 rows) ────────────────────────────────────────────
# Spread across LOW / MID / HIGH amount bands — no label leakage
def sample_amount(band):
    if band == 'low': return round(np.random.uniform(1000,  25000), 2)
    if band == 'mid': return round(np.random.uniform(25000, 75000), 2)
    return                    round(np.random.uniform(75000,200000), 2)

bands = np.random.choice(['low', 'mid', 'high'], size=N_FRAUD, p=[0.30, 0.40, 0.30])
fraud_amounts = np.array([sample_amount(b) for b in bands])

fh = np.array([2,1.5,2.5,3,2,1, 1,1,2,2.5,3,3, 2.5,3,3,2.5,2.5,2, 2,2,2,1.5,1.5,1.5])
fh /= fh.sum()
fraud_hours = np.random.choice(24, size=N_FRAUD, p=fh)

fraud_df = pd.DataFrame({
    'transaction_id':      [str(uuid.uuid4())[:12] for _ in range(N_FRAUD)],
    'timestamp':           make_timestamps(fraud_hours, N_FRAUD),
    'hour_of_day':         fraud_hours,
    'day_of_week':         np.random.randint(0, 7, size=N_FRAUD),
    'month':               np.random.randint(1, 7, size=N_FRAUD),
    'amount_usd':          fraud_amounts,
    'business_type':       np.random.choice(BUSINESS_TYPES, size=N_FRAUD, p=BIZ_WEIGHTS),
    'beneficiary_country': np.random.choice(CORRIDORS,      size=N_FRAUD, p=COR_WEIGHTS),
    'payment_rail':        np.random.choice(RAILS,          size=N_FRAUD, p=RAIL_WEIGHTS),
    'is_new_beneficiary':  np.random.choice([0, 1], size=N_FRAUD, p=[0.40, 0.60]),
    'is_fraud':            1,
    'fraud_type':          np.random.choice(FRAUD_TYPES, size=N_FRAUD),
})

# ── COMBINE, SHUFFLE, SAVE ────────────────────────────────────────────────────
df_practice = pd.concat([legit_df, fraud_df], ignore_index=True)
df_practice = df_practice.sample(frac=1, random_state=42).reset_index(drop=True)
df_practice['timestamp'] = pd.to_datetime(df_practice['timestamp'])

df_practice.to_csv('../data/raw/practice_dataset.csv', index=False)

print(f"Rows:        {len(df_practice):,}")
print(f"Fraud cases: {df_practice['is_fraud'].sum()}")
print(f"Fraud rate:  {df_practice['is_fraud'].mean()*100:.2f}%")
print(f"Columns:     {list(df_practice.columns)}")
print(df_practice.head(3))

---
## Dataset Augmentation — Fix Unrealistic Fraud Patterns

**Appends new rows only. Zero existing records are changed.**

Targets fixed:
- Fraud rate in `$100K+` band reduced to realistic level (was near-100%)
- Fraud rate in `$50K–$200K` band diluted
- Artificial 2–4 AM fraud spike removed
- SME fraud rate brought in line with other business types
- Fraud spread across all hours including business hours

In [ ]:
import pandas as pd
import numpy as np
import uuid
from datetime import datetime, timedelta

np.random.seed(77)

# ── LOAD EXISTING (no changes to these rows) ─────────────────
df_existing = pd.read_csv('../data/raw/practice_dataset.csv', parse_dates=['timestamp'])
print(f"Existing: {len(df_existing):,} rows  |  Fraud: {df_existing['is_fraud'].sum()}  |  Rate: {df_existing['is_fraud'].mean()*100:.2f}%")

# ── DIAGNOSE BEFORE ──────────────────────────────────────────
bins   = [0, 25_000, 50_000, 100_000, 10_000_001]
labels = ['$0–25K', '$25–50K', '$50–100K', '$100K+']
df_existing['_band'] = pd.cut(df_existing['amount_usd'], bins=bins, labels=labels)

print("\n--- Fraud rate by amount band (BEFORE) ---")
b = df_existing.groupby('_band', observed=True)['is_fraud'].agg(fraud='sum', total='count', rate='mean')
b['rate%'] = (b['rate'] * 100).round(1)
print(b[['fraud', 'total', 'rate%']])

print("\n--- Fraud cases by hour 0–5 AM (BEFORE) ---")
print(df_existing[df_existing['hour_of_day'] < 6]
      .groupby('hour_of_day')['is_fraud'].agg(fraud='sum', total='count'))

print("\n--- Fraud rate by business type (BEFORE) ---")
b2 = df_existing.groupby('business_type')['is_fraud'].agg(fraud='sum', total='count', rate='mean')
b2['rate%'] = (b2['rate'] * 100).round(1)
print(b2[['fraud', 'total', 'rate%']].sort_values('rate%', ascending=False))

df_existing = df_existing.drop(columns=['_band'])

# ── SCHEMA ───────────────────────────────────────────────────
BUSINESS_TYPES = ['Exporter', 'Importer', 'IT Services', 'EdTech', 'SaaS', 'SME']
BIZ_WEIGHTS    = [0.25, 0.20, 0.20, 0.15, 0.10, 0.10]
CORRIDORS      = ['USA', 'UK', 'UAE', 'Singapore', 'Germany', 'Australia', 'Canada', 'Japan']
COR_WEIGHTS    = [0.25, 0.20, 0.18, 0.12, 0.08, 0.07, 0.06, 0.04]
RAILS          = ['SWIFT', 'Fedwire', 'CHIPS', 'SEPA', 'CHAPS']
RAIL_WEIGHTS   = [0.60, 0.15, 0.10, 0.10, 0.05]
FRAUD_TYPES    = ['TBML_over_invoicing', 'TBML_under_invoicing', 'round_tripping',
                  'ATO', 'structuring', 'dormant_account', 'purpose_code_abuse']
START_DATE     = datetime(2026, 1, 1)

def make_ts(hours, n):
    days = np.random.randint(0, 180, n)
    mins = np.random.randint(0,  60, n)
    return pd.to_datetime([
        START_DATE + timedelta(days=int(d), hours=int(h), minutes=int(m))
        for d, h, m in zip(days, hours, mins)
    ])

def block(n, hours, amounts, biz=None, is_fraud=0, fraud_labels=None, new_ben_p=0.15):
    biz_col = biz if biz is not None else np.random.choice(BUSINESS_TYPES, n, p=BIZ_WEIGHTS)
    ft = np.random.choice(FRAUD_TYPES, n) if (is_fraud == 1 and fraud_labels is None) else fraud_labels
    return pd.DataFrame({
        'transaction_id':      [str(uuid.uuid4())[:12] for _ in range(n)],
        'timestamp':           make_ts(hours, n),
        'hour_of_day':         hours,
        'day_of_week':         np.random.randint(0, 7, n),
        'month':               np.random.randint(1, 7, n),
        'amount_usd':          amounts,
        'business_type':       biz_col,
        'beneficiary_country': np.random.choice(CORRIDORS, n, p=COR_WEIGHTS),
        'payment_rail':        np.random.choice(RAILS,     n, p=RAIL_WEIGHTS),
        'is_new_beneficiary':  np.random.choice([0, 1], n, p=[1 - new_ben_p, new_ben_p]),
        'is_fraud':            is_fraud,
        'fraud_type':          ft if is_fraud == 1 else np.full(n, np.nan),
    })

batches = []

# BATCH 1 — Legitimate $100K+ (800 rows)
# Why: near-100% fraud rate in this band; need many legit rows to dilute it
n = 800
batches.append(block(
    n, hours=np.random.choice(range(7, 20), n),
    amounts=np.random.uniform(100_000, 500_000, n).round(2),
    new_ben_p=0.10
))

# BATCH 2 — Legitimate $50K–$100K (500 rows)
# Why: fraud disproportionately concentrated in $50K–$200K band
n = 500
batches.append(block(
    n, hours=np.random.choice(range(8, 19), n),
    amounts=np.random.uniform(50_000, 100_000, n).round(2),
    new_ben_p=0.12
))

# BATCH 3 — Legitimate 0–5 AM (400 rows)
# Why: dilute the artificial fraud spike at 2–4 AM
n = 400
batches.append(block(
    n, hours=np.random.choice(range(0, 6), n),
    amounts=np.clip(np.random.lognormal(9.4, 0.7, n), 1_000, 120_000).round(2),
    new_ben_p=0.18
))

# BATCH 4 — Legitimate SME (350 rows)
# Why: SME has the highest fraud rate; add clean SME transactions to normalise it
n = 350
batches.append(block(
    n, hours=np.random.choice(range(8, 19), n),
    amounts=np.clip(np.random.lognormal(9.1, 0.65, n), 1_000, 75_000).round(2),
    biz=np.full(n, 'SME'),
    new_ben_p=0.12
))

# BATCH 5 — Fraud at daytime hours (40 rows)
# Why: fraud should not be artificially absent during 8 AM–8 PM
n = 40
batches.append(block(
    n, hours=np.random.choice(range(8, 20), n),
    amounts=np.random.uniform(5_000, 150_000, n).round(2),
    is_fraud=1, new_ben_p=0.55
))

# BATCH 6 — Fraud in low-amount range $1K–$25K (20 rows)
# Why: fraud should appear across all amount bands, not only mid-to-high
n = 20
batches.append(block(
    n, hours=np.random.choice(range(0, 24), n),
    amounts=np.random.uniform(1_000, 25_000, n).round(2),
    is_fraud=1, new_ben_p=0.50
))

# ── APPEND ONLY — existing rows untouched ────────────────────
df_new = pd.concat(batches, ignore_index=True)
df_augmented = pd.concat([df_existing, df_new], ignore_index=True)
df_augmented.to_csv('../data/raw/practice_dataset.csv', index=False)

# ── VERIFY AFTER ─────────────────────────────────────────────
df_augmented['_band'] = pd.cut(df_augmented['amount_usd'], bins=bins, labels=labels)

print(f"\n=== AFTER AUGMENTATION ===")
print(f"Total rows:  {len(df_augmented):,}  (+{len(df_new):,} added)")
print(f"Total fraud: {df_augmented['is_fraud'].sum()}")
print(f"Fraud rate:  {df_augmented['is_fraud'].mean()*100:.2f}%")

print("\n--- Fraud rate by amount band (AFTER) ---")
a = df_augmented.groupby('_band', observed=True)['is_fraud'].agg(fraud='sum', total='count', rate='mean')
a['rate%'] = (a['rate'] * 100).round(1)
print(a[['fraud', 'total', 'rate%']])

print("\n--- Fraud cases by hour 0–5 AM (AFTER) ---")
print(df_augmented[df_augmented['hour_of_day'] < 6]
      .groupby('hour_of_day')['is_fraud'].agg(fraud='sum', total='count'))

print("\n--- Fraud rate by business type (AFTER) ---")
a2 = df_augmented.groupby('business_type')['is_fraud'].agg(fraud='sum', total='count', rate='mean')
a2['rate%'] = (a2['rate'] * 100).round(1)
print(a2[['fraud', 'total', 'rate%']].sort_values('rate%', ascending=False))